# Mule Pattern Learner — full pipeline (post-load)

Runs every stage AFTER the loading job, in dependency order:

**community detection → entity resolution → features → masking → smoke → train → hidden eval**

Long GSQL queries go through `gsql_install.run_query` (async/detached), so they won't drop the connection on Savanna. Community/ER cells are idempotent (skip if output exists; set `RERUN=True` to force). The `mem()` helper + per-epoch separation are your live health checks.

## 0 · Setup & connect

In [ ]:
import subprocess
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if (ROOT / "src").is_dir() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mule_pattern_learner.tigergraph.client import Client
from mule_pattern_learner.tigergraph.settings import Settings
from mule_pattern_learner.tigergraph import gsql_install

client = Client(Settings())
print("connected:", client.graphname)

In [ ]:
def show(text: str, n: int = 300) -> None:
    text = str(text).strip()
    print(text[:n] + ("\n... [truncated]" if len(text) > n else ""))


def count(vertex_type: str) -> int:
    raw = client.conn.getVertexCount(vertex_type, realtime=True)
    return raw if isinstance(raw, int) else -1


# Flip to True to force recompute of community + entity resolution.
RERUN = False

## M · Memory readout
Call `mem("label")` anywhere. Watch it across the heavy stages — and especially across training epochs — to confirm nothing leaks. This is the real memory check; no short probe can see over-epoch growth.

In [ ]:
def mem(label: str = "") -> None:
    try:
        import psutil

        rss_gb = psutil.Process().memory_info().rss / 1e9
        print(f"  [mem] {label:<24} RSS = {rss_gb:.2f} GB")
    except ImportError:
        print("  [mem] psutil not installed (pip install psutil) — skipping")


mem("session start")

## 1 · Inspect schema (read-only)

In [ ]:
vtypes = client.conn.getVertexTypes()
print("vertex types:", vtypes)
for vt in ("Account", "Party", "Resolved_Entity", "Connected_Component"):
    if vt in vtypes:
        print(f"  {vt:20s} {count(vt)}")

## 2 · Community detection
`weight_account_edges` → `cluster_with_wcc`. Writes `com_id`/`com_size` onto Accounts and materializes `Connected_Component`. Idempotent.

In [ ]:
for key in ("weight_account_edges", "cluster_with_wcc"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key} -> {gsql_install.get_query_from_file(key)}")

existing_components = count("Connected_Component")
if RERUN or existing_components <= 0:
    print("running weight_account_edges (async) ...")
    show(gsql_install.run_query(client, "weight_account_edges"))
    print("running cluster_with_wcc (async) ...")
    show(gsql_install.run_query(client, "cluster_with_wcc", {"min_link_weight": 2}))
    print("Connected_Component now:", count("Connected_Component"))
else:
    print(f"skip: {existing_components} Connected_Component already exist (RERUN=True to redo)")
mem("after community")

## 3 · Entity resolution
`match_parties` (Same_As edges) → `unify_parties` (Resolved_Entity + Party_In_Entity). Feeds the GNN's Party→Resolved_Entity hops. Idempotent.

**Expect FEW Same_As edges** — most synthetic parties are unique, so Resolved_Entity ≈ Party count. That is correct, not a bug.

In [ ]:
for key in ("match_parties", "unify_parties"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key} -> {gsql_install.get_query_from_file(key)}")

existing_entities = count("Resolved_Entity")
if RERUN or existing_entities <= 0:
    print("running match_parties (async) ...")
    show(gsql_install.run_query(client, "match_parties"))
    print("running unify_parties (async) ...")
    show(gsql_install.run_query(client, "unify_parties"))
    print("Resolved_Entity now:", count("Resolved_Entity"))
else:
    print(f"skip: {existing_entities} Resolved_Entity already exist (RERUN=True to redo)")
mem("after entity resolution")

## 4 · Feature queries
Per-account features the GNN reads. Run AFTER community + ER. All async (some slow on 200k). They overwrite attributes, so always safe to re-run.

In [ ]:
_FEATURE_QUERIES = [
    "derive_max_bins",
    "derive_reference_epoch",
    "account_account_degree",
    "identity_sharing",
    "money_flow",
    "pagerank",
    "triangle_clustering",
    "temporal_features",
    "fastrp",
]

for key in _FEATURE_QUERIES:
    print(f"== {key} ==")
    gsql_install.install_query(client, key)
    print(f"   installed -> {gsql_install.get_query_from_file(key)}")
    show(gsql_install.run_query(client, key), 200)
    mem(f"after {key}")

## 5 · Masking (PU labels + split)
CLI script, run via subprocess. Writes pu_label + is_train/is_val/is_test to the graph and a parquet. `0.08` gave val_pos ≈ 73 on the 200k graph.

In [ ]:
_REVEAL_PREVALENCE = "0.08"
mask_cmd = [
    sys.executable,
    "scripts/pipeline/after_load/masking.py",
    "--reveal-prevalence",
    _REVEAL_PREVALENCE,
]
print("running:", " ".join(mask_cmd))
proc = subprocess.run(mask_cmd, capture_output=True, text=True)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-2000:])
    raise RuntimeError("masking failed; see stderr above")
mem("after masking")

## 6 · Smoke test (wiring + timing + memory probe)
Real pipeline, a few batches. Confirms loaders/loss/val/checkpoint, prints per-batch time, and shows the validation `separation` line. **Positive separation = the positive_weight fix is live.** Negative = mis-wired; stop before the long run.

In [ ]:
proc = subprocess.run([sys.executable, "scripts/demos/smoke.py"], capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-2000:])

## 7 · Train
`--estimated-mules 430` (the 200k true count, not 216) sets the nnPU prior. Watch the per-epoch `separation` line — should be POSITIVE and stable. The loop.py retry wrapper survives network blips.

**Run this in a terminal (tmux/nohup), not the notebook** — a multi-day job shouldn't depend on the browser staying open.

In [ ]:
_ESTIMATED_MULES = "430"
train_cmd = [
    sys.executable,
    "-m",
    "mule_pattern_learner.training.train",
    "--estimated-mules",
    _ESTIMATED_MULES,
]
print("run this in a terminal for the long job:")
print("  " + " ".join(train_cmd))
# To run inside the notebook (blocks for the whole run), uncomment:
# subprocess.run(train_cmd)

## 8 · Hidden-mule evaluation — the real verdict
After training, score the checkpoint against the synthetic answer key. AUC over the HIDDEN mules vs the 0.43 baseline is the number that says whether the model generalizes.

In [ ]:
print("run after training completes:")
print("  " + " ".join([sys.executable, "scripts/experiments/evaluate_hidden.py"]))
# proc = subprocess.run([sys.executable, "scripts/experiments/evaluate_hidden.py"], capture_output=True, text=True)
# print(proc.stdout[-3000:])